# Text-to-Shot: Joint Person-Camera Trajectory Generation

## via Dual-Branch Diffusion Model

---

This notebook provides a complete walkthrough of the **Text-to-Shot** project pipeline.

**Project Goal**: Given a natural language text description, simultaneously generate:
1. **Person trajectory** (T, 3) — root position in 3D space
2. **Camera trajectory** (T, 6) — camera position + rotation (tx, ty, tz, azimuth, elevation, roll)

**Key Innovation**: Unlike prior work (E.T./DIRECTOR) which generates camera motion *given* pre-existing character motion, this system generates **both trajectories jointly from text alone**.

---

### Pipeline Overview

```
Step 1: Environment Setup
Step 2: Download E.T. Dataset
Step 3: Preprocess — Extract paired person + camera trajectories
Step 4: Filter to single-person subset
Step 5: Compute normalization statistics
Step 6: Data exploration & visualization
Step 7: Model architecture walkthrough
Step 8: Training
Step 9: Inference / Generation
Step 10: Evaluation
```

---
## Step 1: Environment Setup

Install dependencies and verify the environment.

In [ ]:
# Install project dependencies (run once)
# !pip install torch torchvision transformers matplotlib huggingface_hub scipy numpy tqdm pyyaml einops

In [ ]:
import os
import sys
import json
import yaml
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load project configuration
with open("configs/default.yaml", "r") as f:
    config = yaml.safe_load(f)

print("=" * 60)
print("Project Configuration")
print("=" * 60)
for section, values in config.items():
    print(f"\n[{section}]")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"  {k}: {v}")
    else:
        print(f"  {values}")

---
## Step 2: Download E.T. Dataset

Download the **E.T. (Exceptional Trajectories)** dataset from HuggingFace.

Source: `robin-courant/et-data`

The dataset contains:
- `traj/` — Camera trajectory files (3x4 extrinsic matrices)
- `caption/` — Full scene text descriptions
- `caption_cam/` — Camera-specific descriptions
- `smplh/` — SMPL-H body pose data (includes root translation)
- `char/` — Character data

In [ ]:
# === Configuration ===
# Modify these paths to match your environment
ET_DATA_ROOT = config["data"]["et_data_root"]       # Raw E.T. dataset
STC_DATA_ROOT = config["data"]["data_root"]          # Preprocessed output
CHECKPOINT_DIR = config["paths"]["checkpoint_dir"]    # Model checkpoints
OUTPUT_DIR = config["paths"]["output_dir"]            # Generation outputs

print(f"E.T. data root:    {ET_DATA_ROOT}")
print(f"STC data root:     {STC_DATA_ROOT}")
print(f"Checkpoint dir:    {CHECKPOINT_DIR}")
print(f"Output dir:        {OUTPUT_DIR}")

In [ ]:
# Download E.T. dataset (skip if already exists)
# This is equivalent to: python scripts/download_et_data.py --download-dir <ET_DATA_ROOT>

REQUIRED_DIRS = ["traj", "caption"]

def check_dataset_exists(download_dir):
    if not os.path.isdir(download_dir):
        return False
    return all(os.path.isdir(os.path.join(download_dir, d)) for d in REQUIRED_DIRS)

if check_dataset_exists(ET_DATA_ROOT):
    print(f"[Skip] E.T. dataset already exists at: {ET_DATA_ROOT}")
    for d in ["traj", "caption", "caption_cam", "smplh", "char"]:
        path = os.path.join(ET_DATA_ROOT, d)
        if os.path.isdir(path):
            n = len(os.listdir(path))
            print(f"  [OK] {d}/ ({n} entries)")
        else:
            print(f"  [--] {d}/ (not found)")
else:
    print(f"Downloading E.T. dataset to: {ET_DATA_ROOT}")
    from huggingface_hub import snapshot_download
    os.makedirs(ET_DATA_ROOT, exist_ok=True)
    snapshot_download(
        repo_id="robin-courant/et-data",
        repo_type="dataset",
        local_dir=ET_DATA_ROOT,
        local_dir_use_symlinks=False,
    )
    print(f"Download complete: {ET_DATA_ROOT}")

---
## Step 3: Preprocess E.T. Data

Convert raw E.T. data into the joint training format:

1. **Camera trajectory**: Parse 3x4 extrinsic matrices -> 6D (tx, ty, tz, azimuth, elevation, roll)
2. **Person trajectory**: Extract SMPL-H root translation (or estimate from camera look-at)
3. **Resample** all trajectories to 48 frames (2s @ 24fps)
4. **Classify** camera motion type and shot type from captions
5. **Save** .npy files + train/test index JSON files

In [ ]:
# Core preprocessing functions

def parse_extrinsic_line(line: str) -> np.ndarray:
    """Parse 12 floats into a 3x4 [R|t] matrix."""
    vals = [float(x) for x in line.strip().split()]
    assert len(vals) == 12, f"Expected 12 values, got {len(vals)}"
    return np.array(vals).reshape(3, 4)


def extrinsic_to_6d(R: np.ndarray, t: np.ndarray) -> np.ndarray:
    """Convert (R, t) to 6D: (tx, ty, tz, azimuth, elevation, roll)."""
    tx, ty, tz = t[0], t[1], t[2]
    sy = np.sqrt(R[0, 0] ** 2 + R[1, 0] ** 2)
    if sy > 1e-6:
        elevation = np.arctan2(-R[2, 0], sy)
        azimuth = np.arctan2(R[1, 0], R[0, 0])
        roll = np.arctan2(R[2, 1], R[2, 2])
    else:
        elevation = np.arctan2(-R[2, 0], sy)
        azimuth = np.arctan2(-R[1, 2], R[1, 1])
        roll = 0.0
    return np.array([tx, ty, tz, azimuth, elevation, roll], dtype=np.float32)


def extrinsic_to_lookat(R: np.ndarray, t: np.ndarray, distance: float = 3.0) -> np.ndarray:
    """Estimate look-at point from camera extrinsic (person position proxy)."""
    forward = -R[:, 2]
    forward = forward / (np.linalg.norm(forward) + 1e-8)
    lookat = t + forward * distance
    return lookat.astype(np.float32)


def resample_trajectory(trajectory: np.ndarray, target_frames: int) -> np.ndarray:
    """Linear interpolation to target number of frames."""
    src_frames = trajectory.shape[0]
    if src_frames == target_frames:
        return trajectory
    dim = trajectory.shape[1]
    src_t = np.linspace(0, 1, src_frames)
    tgt_t = np.linspace(0, 1, target_frames)
    resampled = np.zeros((target_frames, dim), dtype=np.float32)
    for d in range(dim):
        resampled[:, d] = np.interp(tgt_t, src_t, trajectory[:, d])
    return resampled


def classify_camera_motion(caption: str) -> str:
    """Classify camera motion type from caption text."""
    text = caption.lower()
    if 'static' in text or 'stationary' in text or 'remains still' in text:
        return 'static'
    if 'push-in' in text or 'push in' in text or 'pushes in' in text:
        return 'dolly-in'
    if 'pull-out' in text or 'pull out' in text or 'pull back' in text:
        return 'dolly-out'
    if 'dolly' in text:
        return 'dolly-in' if ('in' in text or 'forward' in text) else 'dolly-out'
    if 'pan' in text:
        return 'pan-left' if 'left' in text else 'pan-right'
    if 'tilt' in text or 'crane' in text or 'pedestal' in text:
        return 'crane-up' if ('up' in text or 'rise' in text) else 'crane-down'
    if 'orbit' in text or 'arc' in text or 'circular' in text:
        return 'orbit'
    if 'truck' in text or 'lateral' in text or 'tracking' in text or 'follow' in text or 'track' in text:
        return 'track'
    if 'zoom' in text:
        return 'dolly-in' if 'in' in text else 'dolly-out'
    return 'static'


def infer_shot_type(caption: str) -> str:
    """Infer shot type from caption text."""
    text = caption.lower()
    if 'close-up' in text or 'closeup' in text:
        return 'close-up'
    if 'wide' in text or 'establish' in text:
        return 'wide-shot'
    if 'over the shoulder' in text or 'over-the-shoulder' in text:
        return 'over-the-shoulder'
    if 'two-shot' in text or 'two shot' in text:
        return 'two-shot'
    return 'medium-shot'


print("Preprocessing functions loaded.")

# Example: classify some captions
examples = [
    "The camera pushes in slowly toward the character",
    "A wide establishing shot, camera remains static",
    "Camera pans left following the subject",
    "Close-up, the camera orbits around the person",
]
for ex in examples:
    motion = classify_camera_motion(ex)
    shot = infer_shot_type(ex)
    print(f"  [{motion:12s} | {shot:18s}] {ex}")

In [ ]:
# Run the full preprocessing pipeline
# Equivalent to: python scripts/preprocess_et_data.py --et-root <ET_DATA_ROOT> --output-root <STC_DATA_ROOT>

NUM_FRAMES = config["trajectory"]["default_num_frames"]  # 48
MIN_FRAMES = 10
LOOKAT_DISTANCE = 3.0

# Check if already preprocessed
train_index_path = os.path.join(STC_DATA_ROOT, "train_index.json")
if os.path.exists(train_index_path):
    print(f"[Skip] Preprocessed data already exists at: {STC_DATA_ROOT}")
    with open(train_index_path) as f:
        train_idx = json.load(f)
    test_index_path = os.path.join(STC_DATA_ROOT, "test_index.json")
    test_idx = json.load(open(test_index_path)) if os.path.exists(test_index_path) else []
    print(f"  Train: {len(train_idx)} samples")
    print(f"  Test:  {len(test_idx)} samples")
else:
    print("Running full preprocessing... (this may take a few minutes)")
    # Uncomment the line below to run preprocessing:
    # !python scripts/preprocess_et_data.py --et-root {ET_DATA_ROOT} --output-root {STC_DATA_ROOT} --num-frames {NUM_FRAMES}
    print("\nAlternatively, run from command line:")
    print(f"  python scripts/preprocess_et_data.py --et-root {ET_DATA_ROOT} --output-root {STC_DATA_ROOT}")

---
## Step 4: Filter to Single-Person Subset

The dual-branch model trains on single-person scenes. This step filters multi-person samples based on caption keywords.

In [ ]:
# Single-person filtering logic

SINGLE_PERSON_KEYWORDS = [
    "a person", "the person", "the character", "single subject", "one person",
    "the subject", "the main character", "a character", "one character",
    "a man ", "a woman ", "the man ", "the woman ",
]

MULTI_PERSON_KEYWORDS = [
    "two people", "two persons", "both characters", "two characters",
    "two men", "two women", "the two", "they ", "them ",
    "dialogue", "conversation", "between two", "both people",
    "two-shot", "two shot", "over the shoulder",
]


def classify_person_count(text: str) -> str:
    """Classify as single-person, multi-person, or unknown."""
    if not text or not text.strip():
        return "unknown"
    lower = text.lower().strip()
    for kw in MULTI_PERSON_KEYWORDS:
        if kw in lower:
            return "multi"
    for kw in SINGLE_PERSON_KEYWORDS:
        if kw in lower:
            return "single"
    return "unknown"


# Check if filtered index already exists
sp_index_path = os.path.join(STC_DATA_ROOT, "train_index_single_person.json")
if os.path.exists(sp_index_path):
    print(f"[Skip] Single-person index exists: {sp_index_path}")
    with open(sp_index_path) as f:
        sp_train = json.load(f)
    print(f"  Single-person train samples: {len(sp_train)}")
else:
    print("Run filtering:")
    print(f"  python scripts/filter_et_single_person.py --data-root {STC_DATA_ROOT}")
    # Or uncomment:
    # !python scripts/filter_et_single_person.py --data-root {STC_DATA_ROOT}

---
## Step 5: Compute Normalization Statistics

Compute per-dimension mean and standard deviation over all training trajectories. This is used to normalize data during training and denormalize during inference.

In [ ]:
# Check if norm stats exist
norm_stats_path = config["data"].get("norm_stats_path", os.path.join(STC_DATA_ROOT, "norm_stats.json"))

if os.path.exists(norm_stats_path):
    with open(norm_stats_path) as f:
        norm_stats = json.load(f)
    print(f"Normalization statistics loaded from: {norm_stats_path}")
    print(f"  Samples:   {norm_stats.get('n_samples', '?')}")
    print(f"  Total dim: {norm_stats.get('total_dim', '?')}")
    print(f"  Frames:    {norm_stats.get('num_frames', '?')}")
    mean_arr = np.array(norm_stats["mean"])
    std_arr = np.array(norm_stats["std"])
    print(f"  Mean range: [{mean_arr.min():.3f}, {mean_arr.max():.3f}]")
    print(f"  Std  range: [{std_arr.min():.3f}, {std_arr.max():.3f}]")
else:
    print("Norm stats not found. Run:")
    print(f"  python scripts/compute_norm_stats.py --data-root {STC_DATA_ROOT}")
    # Or uncomment:
    # !python scripts/compute_norm_stats.py --data-root {STC_DATA_ROOT}

---
## Step 6: Data Exploration & Visualization

Load the dataset and visualize sample trajectories to understand the data format.

In [ ]:
# Load the dataset
from src.data.dataset import JointTrajectoryDataset, collate_fn
from torch.utils.data import DataLoader

model_cfg = config["model"]
traj_cfg = config["trajectory"]

# Use single-person subset if available
index_file = "train_index_single_person.json"
if not os.path.exists(os.path.join(STC_DATA_ROOT, index_file)):
    index_file = "train_index.json"

dataset = JointTrajectoryDataset(
    data_root=STC_DATA_ROOT,
    split="train",
    num_frames=traj_cfg["default_num_frames"],
    person_dim=model_cfg["person_dim"],
    camera_dim=model_cfg["camera_dim"],
    index_file=index_file,
    norm_stats_path=norm_stats_path if os.path.exists(norm_stats_path) else None,
)

print(f"Dataset loaded: {len(dataset)} samples")
print(f"  Index file: {index_file}")
print(f"  Num frames: {traj_cfg['default_num_frames']}")
print(f"  Person dim: {model_cfg['person_dim']} -> flat: {model_cfg['person_dim'] * traj_cfg['default_num_frames']}")
print(f"  Camera dim: {model_cfg['camera_dim']} -> flat: {model_cfg['camera_dim'] * traj_cfg['default_num_frames']}")
print(f"  Total dim:  {dataset.total_dim}")

In [ ]:
# Inspect a single sample
if len(dataset) > 0:
    sample = dataset[0]
    print(f"Sample keys: {list(sample.keys())}")
    print(f"  y shape:      {sample['y'].shape}  (joint trajectory vector)")
    print(f"  text:         {sample['text'][:100]}...")
    print(f"  shot_type:    {sample['shot_type']}")
    print(f"  motion_type:  {sample['motion_type']}")
    print(f"  sample_id:    {sample['sample_id']}")
    
    # Decompose the joint vector
    T = traj_cfg["default_num_frames"]
    p_dim = model_cfg["person_dim"]
    c_dim = model_cfg["camera_dim"]
    person_total = T * p_dim
    
    y = sample['y']
    # If normalized, denormalize for visualization
    if dataset.norm_mean is not None:
        y = y * dataset.norm_std + dataset.norm_mean
    
    person_traj = y[:person_total].numpy().reshape(T, p_dim)
    camera_traj = y[person_total:].numpy().reshape(T, c_dim)
    
    print(f"\n  Person trajectory: shape={person_traj.shape}, range=[{person_traj.min():.2f}, {person_traj.max():.2f}]")
    print(f"  Camera trajectory: shape={camera_traj.shape}, range=[{camera_traj.min():.2f}, {camera_traj.max():.2f}]")

In [ ]:
# Visualize sample trajectories
def visualize_sample(person_traj, camera_traj, text="", figsize=(16, 6)):
    """3-panel visualization: 3D paths, camera params, person position."""
    fig = plt.figure(figsize=figsize, facecolor='#1a1a2e')

    # Panel 1: 3D Trajectories
    ax1 = fig.add_subplot(131, projection='3d', facecolor='#1a1a2e')
    ax1.plot3D(camera_traj[:, 0], camera_traj[:, 1], camera_traj[:, 2],
               color='#FFE66D', linewidth=2, label='Camera')
    ax1.plot3D(person_traj[:, 0], person_traj[:, 1], person_traj[:, 2],
               color='#4ECDC4', linewidth=2, label='Person')
    ax1.scatter(*camera_traj[0, :3], color='#FFE66D', s=60, marker='o', edgecolors='white')
    ax1.scatter(*person_traj[0], color='#4ECDC4', s=60, marker='^', edgecolors='white')
    ax1.set_title('3D Trajectories', color='white', fontsize=11)
    ax1.legend(fontsize=8, labelcolor='white', framealpha=0.3)
    ax1.tick_params(colors='gray', labelsize=7)

    # Panel 2: Camera Parameters
    ax2 = fig.add_subplot(132, facecolor='#2C3E50')
    t = np.linspace(0, 1, len(camera_traj))
    names = ['tx', 'ty', 'tz', 'az', 'el', 'roll']
    colors = ['#FF6B6B', '#FFE66D', '#4ECDC4', '#C44ECD', '#95E66D', '#FF9F43']
    for i, (name, c) in enumerate(zip(names, colors)):
        ax2.plot(t, camera_traj[:, i], color=c, linewidth=1.5, label=name, alpha=0.8)
    ax2.set_title('Camera Parameters', color='white', fontsize=11)
    ax2.legend(fontsize=7, labelcolor='white', framealpha=0.3, ncol=2)
    ax2.tick_params(colors='gray', labelsize=7)
    ax2.grid(alpha=0.15)

    # Panel 3: Person Position
    ax3 = fig.add_subplot(133, facecolor='#2C3E50')
    pnames = ['px', 'py', 'pz']
    pcolors = ['#4ECDC4', '#95E66D', '#C44ECD']
    for i, (name, c) in enumerate(zip(pnames, pcolors)):
        ax3.plot(t, person_traj[:, i], color=c, linewidth=1.5, label=name)
    ax3.set_title('Person Position', color='white', fontsize=11)
    ax3.legend(fontsize=8, labelcolor='white', framealpha=0.3)
    ax3.tick_params(colors='gray', labelsize=7)
    ax3.grid(alpha=0.15)

    if text:
        fig.suptitle(f'"{text[:80]}"', color='white', fontsize=10, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()


# Visualize the first sample
if len(dataset) > 0:
    visualize_sample(person_traj, camera_traj, text=sample['text'])

In [ ]:
# Dataset statistics: shot type and motion type distributions
if len(dataset) > 0:
    shot_counts = {}
    motion_counts = {}
    inv_shot = {v: k for k, v in dataset.SHOT_TYPE_MAP.items()}
    inv_motion = {v: k for k, v in dataset.MOTION_TYPE_MAP.items()}
    
    for i in range(min(len(dataset), 500)):  # sample up to 500
        s = dataset[i]
        st = inv_shot.get(s['shot_type'], 'unknown')
        mt = inv_motion.get(s['motion_type'], 'unknown')
        shot_counts[st] = shot_counts.get(st, 0) + 1
        motion_counts[mt] = motion_counts.get(mt, 0) + 1
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.bar(shot_counts.keys(), shot_counts.values(), color='#4ECDC4')
    ax1.set_title('Shot Type Distribution')
    ax1.tick_params(axis='x', rotation=30)
    
    ax2.bar(motion_counts.keys(), motion_counts.values(), color='#FFE66D')
    ax2.set_title('Camera Motion Type Distribution')
    ax2.tick_params(axis='x', rotation=30)
    
    plt.tight_layout()
    plt.show()

---
## Step 7: Model Architecture

The model consists of three main components:

### 7.1 CLIP Text Encoder
Frozen `openai/clip-vit-base-patch32` encodes text descriptions into 512D embeddings.

### 7.2 Dual-Branch Denoiser (JointTrajectoryDenoiser)
A Transformer with separate person/camera branches connected by cross-attention:
- **Person branch**: Self-attention (temporal) + cross-attention (to camera) + FiLM FFN
- **Camera branch**: Self-attention (temporal) + cross-attention (to person) + FiLM FFN
- Conditioned on: text (512D) + timestep (128D) + shot type (64D) + motion type (64D)

### 7.3 Gaussian Diffusion (DDPM)
- 1000 timesteps, cosine beta schedule
- Forward process: progressively add noise
- Reverse process: iteratively denoise with Classifier-Free Guidance

### FiLM Layer
Feature-wise Linear Modulation: `output = gamma * input + beta` where gamma/beta are predicted from conditioning.

In [ ]:
# Build and inspect the model
from src.models.denoiser import JointTrajectoryDenoiser
from src.models.diffusion import GaussianDiffusion

device = "cuda" if torch.cuda.is_available() else "cpu"

# Build denoiser
denoiser = JointTrajectoryDenoiser(
    person_dim=model_cfg["person_dim"],       # 3
    camera_dim=model_cfg["camera_dim"],        # 6
    num_frames=traj_cfg["default_num_frames"], # 48
    hidden_dim=model_cfg["hidden_dim"],        # 256
    num_layers=model_cfg["num_layers"],        # 6
    num_heads=model_cfg["num_heads"],          # 4
    text_dim=512,
    timestep_dim=128,
    num_shot_types=len(config["shot_types"]["categories"]),   # 5
    shot_type_dim=config["shot_types"]["embedding_dim"],      # 64
    num_motion_types=len(traj_cfg["motion_types"]),            # 9
    motion_type_dim=traj_cfg.get("motion_type_dim", 64),      # 64
    dropout=model_cfg.get("dropout", 0.1),
).to(device)

# Build diffusion wrapper
diffusion = GaussianDiffusion(
    denoiser=denoiser,
    num_timesteps=config["diffusion"]["num_timesteps"],  # 1000
    beta_schedule=config["diffusion"]["beta_schedule"],  # cosine
).to(device)

# Model statistics
total_params = sum(p.numel() for p in diffusion.parameters())
trainable_params = sum(p.numel() for p in diffusion.parameters() if p.requires_grad)

T = traj_cfg["default_num_frames"]
p_dim = model_cfg["person_dim"]
c_dim = model_cfg["camera_dim"]

print("=" * 60)
print("Model Architecture Summary")
print("=" * 60)
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Joint dimension:      person ({T}x{p_dim}={T*p_dim}) + camera ({T}x{c_dim}={T*c_dim}) = {T*(p_dim+c_dim)}")
print(f"  Hidden dimension:     {model_cfg['hidden_dim']}")
print(f"  Dual-branch layers:   {model_cfg['num_layers']}")
print(f"  Attention heads:      {model_cfg['num_heads']}")
print(f"  Diffusion timesteps:  {config['diffusion']['num_timesteps']}")
print(f"  Beta schedule:        {config['diffusion']['beta_schedule']}")
print(f"  Device:               {device}")
print("=" * 60)

In [ ]:
# Test forward pass with dummy data
batch_size = 2
total_dim = T * p_dim + T * c_dim  # 48*3 + 48*6 = 432

# Dummy inputs
y_noisy = torch.randn(batch_size, total_dim, device=device)
t_step = torch.randint(0, 1000, (batch_size,), device=device)
text_embed = torch.randn(batch_size, 512, device=device)
shot_type = torch.tensor([1, 2], device=device)  # medium-shot, wide-shot
motion_type = torch.tensor([0, 1], device=device)  # static, dolly-in

# Forward pass through denoiser
with torch.no_grad():
    y_pred = denoiser(y_noisy, t_step, text_embed,
                      shot_type=shot_type, motion_type=motion_type)

print(f"Input shape:  {y_noisy.shape}")
print(f"Output shape: {y_pred.shape}")
print(f"Match: {y_noisy.shape == y_pred.shape}")

# Forward pass through diffusion (training loss)
y_clean = torch.randn(batch_size, total_dim, device=device)
loss = diffusion.p_losses(y_clean, text_embed,
                          shot_type=shot_type, motion_type=motion_type)
print(f"\nTraining loss (random init): {loss.item():.4f}")

In [ ]:
# Visualize the diffusion noise schedule
betas = diffusion.betas.cpu().numpy()
alphas_cumprod = diffusion.alphas_cumprod.cpu().numpy()
sqrt_alphas = diffusion.sqrt_alphas_cumprod.cpu().numpy()
sqrt_one_minus = diffusion.sqrt_one_minus_alphas_cumprod.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(betas, color='#FF6B6B', linewidth=1.5)
axes[0].set_title('Beta Schedule (Cosine)')
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Beta')
axes[0].grid(alpha=0.3)

axes[1].plot(alphas_cumprod, color='#4ECDC4', linewidth=1.5)
axes[1].set_title('Cumulative Alpha')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('alpha_bar')
axes[1].grid(alpha=0.3)

axes[2].plot(sqrt_alphas, color='#FFE66D', linewidth=1.5, label='sqrt(alpha_bar)')
axes[2].plot(sqrt_one_minus, color='#C44ECD', linewidth=1.5, label='sqrt(1-alpha_bar)')
axes[2].set_title('Signal vs Noise')
axes[2].set_xlabel('Timestep')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Step 8: Training

Train the dual-branch diffusion model.

**Training Configuration:**
- Optimizer: AdamW (lr=1e-4, weight_decay=1e-4)
- Epochs: 500
- Batch size: 64
- LR schedule: Cosine with 10-epoch warmup
- Gradient clipping: 1.0
- CFG dropout: 10% (randomly zero text embeddings)

**Training Loss:**
$$L = \mathbb{E}_{t, y_0, \epsilon}\left[\|y_0 - f_\theta(y_t, t, c)\|^2\right]$$

where $y_t$ is the noisy joint trajectory at timestep $t$, and $c$ is the conditioning (text + shot type + motion type).

In [ ]:
# Option A: Run training from command line (recommended for full training)
print("Full training command:")
print(f"  python train.py --config configs/default.yaml --device cuda --single-person")
print()
print("Without CLIP (faster, uses random embeddings):")
print(f"  python train.py --config configs/default.yaml --device cuda --single-person --no-clip")
print()
print("Resume from checkpoint:")
print(f"  python train.py --config configs/default.yaml --device cuda --resume {CHECKPOINT_DIR}/stc_epoch50.pth")

In [ ]:
# ============================================================
# Training Hyperparameters (modify here before running)
# ============================================================

NUM_EPOCHS = 500          # <-- Total training epochs, change as needed
BATCH_SIZE = 64           # <-- Batch size
LEARNING_RATE = 1e-4      # <-- Learning rate
WEIGHT_DECAY = 1e-4       # <-- Weight decay
GRADIENT_CLIP = 1.0       # <-- Gradient clipping max norm
CFG_DROPOUT_PROB = 0.1    # <-- Classifier-Free Guidance dropout
SAVE_INTERVAL = 50        # <-- Save checkpoint every N epochs
EVAL_INTERVAL = 10        # <-- Run validation every N epochs
USE_CLIP = True           # <-- True: use CLIP text encoder; False: random embeddings

print(f"Training config:")
print(f"  Epochs:           {NUM_EPOCHS}")
print(f"  Batch size:       {BATCH_SIZE}")
print(f"  Learning rate:    {LEARNING_RATE}")
print(f"  Gradient clip:    {GRADIENT_CLIP}")
print(f"  CFG dropout:      {CFG_DROPOUT_PROB}")
print(f"  Save interval:    {SAVE_INTERVAL}")
print(f"  Eval interval:    {EVAL_INTERVAL}")
print(f"  Use CLIP:         {USE_CLIP}")

In [ ]:
# ============================================================
# Training Loop - run this cell to start training
# ============================================================

# Text encoder
text_encoder = None
if USE_CLIP:
    try:
        from src.models.text_encoder import CLIPTextEncoder
        text_encoder = CLIPTextEncoder(
            model_name=config['text_encoder']['model_name'], device=device
        ).to(device)
        print("CLIP text encoder loaded.")
    except Exception as e:
        print(f"CLIP unavailable ({e}), using random embeddings.")
else:
    print("Using random text embeddings (USE_CLIP=False).")

# Validation dataset
val_index_file = "test_index_single_person.json"
if not os.path.exists(os.path.join(STC_DATA_ROOT, val_index_file)):
    val_index_file = "test_index.json"

val_dataset = JointTrajectoryDataset(
    data_root=STC_DATA_ROOT,
    split="test",
    num_frames=config["trajectory"]["default_num_frames"],
    person_dim=config["model"]["person_dim"],
    camera_dim=config["model"]["camera_dim"],
    index_file=val_index_file,
    norm_stats_path=norm_stats_path if os.path.exists(norm_stats_path) else None,
)

# DataLoaders
train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

# Optimizer
optimizer = torch.optim.AdamW(
    diffusion.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Checkpoint directory
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
train_losses = []
val_losses = []

print(f"\nStarting training for {NUM_EPOCHS} epochs...")
print(f"  Train: {len(dataset)} samples | Val: {len(val_dataset)} samples | Batch: {BATCH_SIZE}")
print(f"  Val evaluated every {EVAL_INTERVAL} epochs")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print()

for epoch in range(NUM_EPOCHS):
    # ---- Train ----
    diffusion.train()
    total_loss = 0
    num_batches = 0

    for batch in train_loader:
        y = batch['y'].to(device)

        if text_encoder is not None:
            text_embed = text_encoder(batch['texts'])
        else:
            text_embed = torch.randn(y.shape[0], 512, device=device)

        if CFG_DROPOUT_PROB > 0:
            drop_mask = torch.rand(y.shape[0], device=device) < CFG_DROPOUT_PROB
            text_embed = text_embed.clone()
            text_embed[drop_mask] = 0.0

        shot_types = batch['shot_types'].to(device)
        shot_type = shot_types if (shot_types >= 0).all() else None
        motion_types = batch['motion_types'].to(device)
        motion_type = motion_types if (motion_types >= 0).all() else None

        loss = diffusion.p_losses(y, text_embed,
                                  shot_type=shot_type, motion_type=motion_type)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(diffusion.parameters(), GRADIENT_CLIP)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    avg_train_loss = total_loss / max(num_batches, 1)
    train_losses.append(avg_train_loss)

    # ---- Validation ----
    avg_val_loss = float('nan')
    if len(val_dataset) > 0 and (epoch + 1) % EVAL_INTERVAL == 0:
        diffusion.eval()
        val_total = 0
        val_batches = 0
        with torch.no_grad():
            for batch in val_loader:
                y = batch['y'].to(device)

                if text_encoder is not None:
                    text_embed = text_encoder(batch['texts'])
                else:
                    text_embed = torch.randn(y.shape[0], 512, device=device)

                shot_types = batch['shot_types'].to(device)
                shot_type = shot_types if (shot_types >= 0).all() else None
                motion_types = batch['motion_types'].to(device)
                motion_type = motion_types if (motion_types >= 0).all() else None

                loss = diffusion.p_losses(y, text_embed,
                                          shot_type=shot_type, motion_type=motion_type)
                val_total += loss.item()
                val_batches += 1

        avg_val_loss = val_total / max(val_batches, 1)

    val_losses.append(avg_val_loss)

    if np.isnan(avg_val_loss):
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Train: {avg_train_loss:.6f}")
    else:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Train: {avg_train_loss:.6f}  Val: {avg_val_loss:.6f}")

    # Save checkpoint
    if (epoch + 1) % SAVE_INTERVAL == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'stc_epoch{epoch+1}.pth')
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': diffusion.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'config': config,
        }, ckpt_path)
        print(f"  Saved: {ckpt_path}")

# Save final model
final_path = os.path.join(CHECKPOINT_DIR, 'stc_final.pth')
torch.save({
    'epoch': NUM_EPOCHS,
    'model_state_dict': diffusion.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': avg_train_loss,
    'val_loss': avg_val_loss,
    'train_losses': train_losses,
    'val_losses': val_losses,
    'config': config,
}, final_path)
print(f"\nTraining complete! Final model: {final_path}")

In [ ]:
# ============================================================
# Visualize Training & Validation Loss Curves
# ============================================================
# Option A: Plot from in-memory lists (run right after training)
# Option B: Load from a saved checkpoint (if restarting the kernel)

# --- Option B: load from checkpoint (uncomment if needed) ---
# ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'stc_final.pth'), map_location='cpu', weights_only=False)
# train_losses = ckpt['train_losses']
# val_losses = ckpt['val_losses']

epochs = list(range(1, len(train_losses) + 1))
val_epochs = [e for e, v in zip(epochs, val_losses) if not np.isnan(v)]
val_vals   = [v for v in val_losses if not np.isnan(v)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# --- Left: full loss curve ---
ax1.plot(epochs, train_losses, color='#FF6B6B', linewidth=1, alpha=0.8, label='Train')
if val_vals:
    ax1.plot(val_epochs, val_vals, color='#4ECDC4', linewidth=2,
             marker='o', markersize=3, label=f'Val (every {EVAL_INTERVAL} ep)')
    best_idx = int(np.argmin(val_vals))
    ax1.annotate(f'Best val: {val_vals[best_idx]:.4f} (ep {val_epochs[best_idx]})',
                 xy=(val_epochs[best_idx], val_vals[best_idx]),
                 xytext=(val_epochs[best_idx] + len(epochs)*0.05,
                         val_vals[best_idx] + (max(train_losses[:50]) - min(train_losses)) * 0.05),
                 arrowprops=dict(arrowstyle='->', color='#4ECDC4'),
                 fontsize=9, color='#4ECDC4')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (MSE)')
ax1.set_title('Training & Validation Loss')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# --- Right: zoomed-in on the last 50% of training ---
mid = len(epochs) // 2
ax2.plot(epochs[mid:], train_losses[mid:], color='#FF6B6B', linewidth=1.2, alpha=0.8, label='Train')
late_val_epochs = [e for e in val_epochs if e > mid]
late_val_vals   = [v for e, v in zip(val_epochs, val_vals) if e > mid]
if late_val_vals:
    ax2.plot(late_val_epochs, late_val_vals, color='#4ECDC4', linewidth=2,
             marker='o', markersize=4, label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (MSE)')
ax2.set_title(f'Loss (last {len(epochs) - mid} epochs, zoomed)')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'loss_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Final train loss: {train_losses[-1]:.6f}")
if val_vals:
    print(f"Final val loss:   {val_vals[-1]:.6f}")
    print(f"Best val loss:    {val_vals[best_idx]:.6f} (epoch {val_epochs[best_idx]})")

---
## Step 9: Inference / Generation

Generate joint person-camera trajectories from text prompts.

**Process:**
1. Encode text with CLIP -> 512D embedding
2. Start from pure noise y_T ~ N(0, I)
3. Iteratively denoise (1000 steps) with Classifier-Free Guidance
4. Denormalize and split into person (T,3) + camera (T,6)
5. Visualize and save results

In [ ]:
# Generation from command line
print("Generate from command line:")
print()
print("# Recommended: DDIM sampling (fast, smooth, deterministic)")
print(f'  python generate.py --checkpoint {CHECKPOINT_DIR}/stc_final.pth --text "A person walks toward camera" --ddim')
print(f'  python generate.py --checkpoint {CHECKPOINT_DIR}/stc_final.pth --text "Close-up shot" --motion dolly-in --ddim --ddim-steps 50')
print(f'  python generate.py --checkpoint {CHECKPOINT_DIR}/stc_final.pth --text "Wide establishing shot" --ddim --guidance-scale 5.0')
print()
print("# Without smoothing (raw model output)")
print(f'  python generate.py --checkpoint {CHECKPOINT_DIR}/stc_final.pth --text "Person walks" --ddim --no-smooth')
print()
print("# Original DDPM sampling (slower, noisier)")
print(f'  python generate.py --checkpoint {CHECKPOINT_DIR}/stc_final.pth --text "A person walks toward camera"')

In [ ]:
# Interactive generation in notebook

from scipy.signal import savgol_filter

SHOT_TYPE_MAP = {
    "close-up": 0, "medium-shot": 1, "wide-shot": 2,
    "over-the-shoulder": 3, "two-shot": 4,
}
MOTION_TYPE_MAP = {
    "static": 0, "dolly-in": 1, "dolly-out": 2,
    "pan-left": 3, "pan-right": 4, "crane-up": 5,
    "crane-down": 6, "track": 7, "orbit": 8,
}


def smooth_trajectory(traj, window=7, polyorder=2):
    """Savitzky-Golay filter to remove high-frequency jitter."""
    if traj.shape[0] < window:
        return traj
    smoothed = np.zeros_like(traj)
    for d in range(traj.shape[1]):
        smoothed[:, d] = savgol_filter(traj[:, d], window_length=window,
                                       polyorder=polyorder)
    return smoothed


def generate_trajectory(diffusion, text, shot_type_name="medium-shot",
                        motion_name="static", guidance_scale=3.0,
                        text_encoder=None, device="cpu", norm_stats=None,
                        use_ddim=True, ddim_steps=50, smooth=True):
    """Generate and visualize a trajectory from text."""
    diffusion.eval()
    
    # Encode text
    if text_encoder is not None:
        text_embed = text_encoder([text])
    else:
        text_embed = torch.randn(1, 512, device=device)
    
    shot_idx = SHOT_TYPE_MAP.get(shot_type_name, 1)
    motion_idx = MOTION_TYPE_MAP.get(motion_name, 0)
    shot_type = torch.tensor([shot_idx], device=device)
    motion_type = torch.tensor([motion_idx], device=device)
    
    sampler = f"DDIM({ddim_steps})" if use_ddim else "DDPM(1000)"
    print(f'Generating: "{text}"')
    print(f'  shot={shot_type_name}, motion={motion_name}, guidance={guidance_scale}, sampler={sampler}')
    
    with torch.no_grad():
        y = diffusion.sample(text_embed, shot_type=shot_type,
                             motion_type=motion_type, device=device,
                             guidance_scale=guidance_scale,
                             use_ddim=use_ddim, ddim_steps=ddim_steps)
    
    # Denormalize
    if norm_stats is not None:
        norm_mean = torch.tensor(norm_stats['mean'], dtype=torch.float32, device=device)
        norm_std = torch.tensor(norm_stats['std'], dtype=torch.float32, device=device)
        y = y * norm_std + norm_mean
    
    y_np = y[0].cpu().numpy()
    T = config['trajectory']['default_num_frames']
    p_dim = config['model']['person_dim']
    c_dim = config['model']['camera_dim']
    person_total = T * p_dim
    
    person_traj = y_np[:person_total].reshape(T, p_dim)
    camera_traj = y_np[person_total:].reshape(T, c_dim)
    
    if smooth:
        person_traj = smooth_trajectory(person_traj)
        camera_traj = smooth_trajectory(camera_traj)
    
    visualize_sample(person_traj, camera_traj, text=f"{motion_name} | {text}")
    return person_traj, camera_traj


print("Generation function ready.")
print("\nTo generate, load a trained checkpoint first, then call:")
print('  person, camera = generate_trajectory(diffusion, "A person walks forward",')
print('                                       motion_name="dolly-in", use_ddim=True)')

In [ ]:
# Load a trained checkpoint and generate
# Uncomment and modify the checkpoint path

"""
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/stc_final.pth"

if os.path.exists(CHECKPOINT_PATH):
    # Load checkpoint
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    diffusion.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded checkpoint: {CHECKPOINT_PATH} (epoch {ckpt.get('epoch', '?')})")
    
    # Load norm stats
    ns = None
    if os.path.exists(norm_stats_path):
        with open(norm_stats_path) as f:
            ns = json.load(f)
    
    # Generate examples
    prompts = [
        ("A person walks toward the camera", "medium-shot", "dolly-out"),
        ("The character stands still in a wide shot", "wide-shot", "static"),
        ("Close-up of person turning, camera orbits", "close-up", "orbit"),
        ("Person runs left, camera tracks", "medium-shot", "track"),
    ]
    
    for text, shot, motion in prompts:
        person, camera = generate_trajectory(
            diffusion, text,
            shot_type_name=shot,
            motion_name=motion,
            guidance_scale=3.0,
            device=device,
            norm_stats=ns,
        )
        print()
else:
    print(f"Checkpoint not found: {CHECKPOINT_PATH}")
    print("Train the model first (Step 8).")
"""

print("(Generation code is commented out — uncomment after training)")

---
## Step 10: Evaluation

Quantitative evaluation metrics:

| Metric | Description |
|--------|-------------|
| **Person MSE/MAE** | Reconstruction error for person trajectory |
| **Camera MSE/MAE** | Reconstruction error for camera trajectory |
| **Person Jerk** | Smoothness of person motion (lower = smoother) |
| **Camera Jerk** | Smoothness of camera motion (lower = smoother) |
| **Path Length** | Total distance traveled |
| **Person-Camera Distance** | Average distance between person and camera (coordination) |

In [ ]:
# Evaluation from command line
print("Run evaluation:")
print(f"  python evaluate.py --checkpoint {CHECKPOINT_DIR}/stc_final.pth --device cuda")
print(f"  python evaluate.py --checkpoint {CHECKPOINT_DIR}/stc_final.pth --device cuda --single-person")

In [ ]:
# Evaluation metrics functions

def compute_smoothness(trajectory: np.ndarray) -> dict:
    """Compute smoothness metrics (jerk, path length)."""
    velocity = np.diff(trajectory, axis=0)
    acceleration = np.diff(velocity, axis=0)
    jerk = np.diff(acceleration, axis=0)
    return {
        'mean_jerk': float(np.mean(np.linalg.norm(jerk, axis=1))) if len(jerk) > 0 else 0.0,
        'path_length': float(np.sum(np.linalg.norm(velocity, axis=1))),
    }


def compute_metrics(gen_person, gen_camera, gt_person, gt_camera):
    """Compute per-sample metrics."""
    person_mse = float(np.mean((gen_person - gt_person) ** 2))
    person_mae = float(np.mean(np.abs(gen_person - gt_person)))
    camera_mse = float(np.mean((gen_camera - gt_camera) ** 2))
    camera_mae = float(np.mean(np.abs(gen_camera - gt_camera)))

    person_smooth = compute_smoothness(gen_person)
    camera_smooth = compute_smoothness(gen_camera)

    cam_pos = gen_camera[:, :3]
    person_cam_dist = np.mean(np.linalg.norm(gen_person - cam_pos, axis=1))

    return {
        'person_mse': person_mse,
        'person_mae': person_mae,
        'camera_mse': camera_mse,
        'camera_mae': camera_mae,
        'person_jerk': person_smooth['mean_jerk'],
        'camera_jerk': camera_smooth['mean_jerk'],
        'person_path_length': person_smooth['path_length'],
        'camera_path_length': camera_smooth['path_length'],
        'person_cam_dist': float(person_cam_dist),
    }


print("Evaluation functions ready.")

In [ ]:
# Load and display evaluation results (if available)
eval_results_path = os.path.join(OUTPUT_DIR, "evaluation_results.json")

if os.path.exists(eval_results_path):
    with open(eval_results_path) as f:
        results = json.load(f)
    
    print("=" * 60)
    print("Evaluation Results")
    print("=" * 60)
    for key, val in results.items():
        if isinstance(val, dict):
            print(f"  {key:25s}: {val['mean']:.6f} +/- {val['std']:.6f}")
        else:
            print(f"  {key:25s}: {val}")
    print("=" * 60)
else:
    print(f"No evaluation results found at: {eval_results_path}")
    print("Run evaluation first (see command above).")

---
## Appendix: Utility Functions

### A.1 Rotation Representations

6D continuous rotation representation (Zhou et al., CVPR 2019):
- More stable than Euler angles for neural networks
- First two columns of rotation matrix, flattened to 6D
- Reconstruct via Gram-Schmidt orthogonalization

### A.2 Toric Camera Parameterization

Places camera on a torus around two subjects (for two-character scenes):
- Parameters: theta (azimuth), phi (elevation), screen positions of both characters
- Packed as 6D vector: `[pA_x, pA_y, pB_x, pB_y, theta, phi]`

In [ ]:
# Rotation utilities demo
from src.utils.smpl_utils import rotation_matrix_to_6d, rotation_6d_to_matrix

# Create a sample rotation matrix (30 degrees around Y axis)
angle = np.radians(30)
R = np.array([
    [np.cos(angle), 0, np.sin(angle)],
    [0, 1, 0],
    [-np.sin(angle), 0, np.cos(angle)],
])

# Convert to 6D and back
r6d = rotation_matrix_to_6d(R)
R_recovered = rotation_6d_to_matrix(r6d)

print("Original R:")
print(R)
print(f"\n6D representation: {r6d}")
print(f"\nRecovered R:")
print(R_recovered)
print(f"\nReconstruction error: {np.abs(R - R_recovered).max():.2e}")

In [ ]:
# Toric camera demo
from src.utils.toric import toric_to_camera_extrinsics, pack_toric_state, unpack_toric_state

# Two characters at different positions
head_A = np.array([1.0, 1.7, 0.0])  # Character A head position
head_B = np.array([-1.0, 1.7, 0.0])  # Character B head position

# Camera on torus: different angles
thetas = np.linspace(0, 2*np.pi, 8, endpoint=False)
camera_positions = []

for theta in thetas:
    R, t = toric_to_camera_extrinsics(
        theta=theta, phi=0.3,
        p_A=np.array([0.3, 0.5]), p_B=np.array([0.7, 0.5]),
        head_A_3d=head_A, head_B_3d=head_B,
    )
    camera_positions.append(t)

camera_positions = np.array(camera_positions)

# Visualize toric camera positions
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(camera_positions[:, 0], camera_positions[:, 1], camera_positions[:, 2],
        'o-', color='#FFE66D', markersize=8, label='Camera positions')
ax.scatter(*head_A, color='#4ECDC4', s=100, marker='^', label='Character A')
ax.scatter(*head_B, color='#FF6B6B', s=100, marker='^', label='Character B')
ax.set_title('Toric Camera Positions (varying theta)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Project Structure Reference

```
Text-to-Shot/
|
|- configs/
|   |- default.yaml              # All model, training, and data configs
|
|- src/
|   |- models/
|   |   |- denoiser.py           # JointTrajectoryDenoiser (dual-branch Transformer)
|   |   |- diffusion.py          # GaussianDiffusion (DDPM with CFG)
|   |   |- text_encoder.py       # CLIPTextEncoder (frozen CLIP)
|   |   |- film.py               # FiLM conditioning layer
|   |- data/
|   |   |- dataset.py            # JointTrajectoryDataset + collate_fn
|   |- utils/
|       |- smpl_utils.py         # Rotation & trajectory utilities
|       |- toric.py              # Toric camera parameterization
|
|- scripts/
|   |- download_et_data.py       # Step 2: Download E.T. dataset
|   |- preprocess_et_data.py     # Step 3: Preprocess to joint format
|   |- filter_et_single_person.py # Step 4: Filter to single-person
|   |- compute_norm_stats.py     # Step 5: Compute normalization stats
|
|- train.py                      # Step 8: Training script
|- generate.py                   # Step 9: Inference / generation
|- evaluate.py                   # Step 10: Quantitative evaluation
|- pyproject.toml                # Dependencies
```

---

**Key Technical Concepts:**

1. **Dual-Branch Diffusion** — Person and camera trajectories are generated jointly with cross-attention ensuring coordination
2. **DDPM** — Iterative refinement from noise over 1000 steps
3. **Classifier-Free Guidance (CFG)** — Improves text-trajectory alignment by interpolating conditioned/unconditioned outputs
4. **FiLM Conditioning** — Injects text/timestep/shot-type into transformer via affine modulation
5. **SMPL-H** — Human body model; uses root translation as person trajectory
6. **Toric Space** — Camera parameterization relative to two subjects on a torus